In [0]:
%sql
USE CATALOG alpine;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS alpine.bronze.dim_entity;

COPY INTO alpine.bronze.dim_entity
FROM (
    SELECT *,
          _metadata.file_path AS source_file,
          current_timestamp() as ingested_at
    FROM '/Volumes/alpine/raw/data/'
)
FILEFORMAT = csv
PATTERN = 'dim_entity*.csv'
FORMAT_OPTIONS('header'='true', 'inferSchema'='true')
COPY_OPTIONS('mergeSchema'='true');

In [0]:
SOURCE = "/Volumes/alpine/raw/data/"   # folder with all the CSVs
TARGET = "alpine.bronze"               # schema we created in step 1

# Money columns per table: cast from double to DECIMAL(18,2)
# so financial totals stay exact to the cent.
# Tables not listed here have no money columns.
money_cols = {
    "fact_gl_actuals":   ["amount_eur"],
    "fact_plan":         ["amount_eur"],
    "fact_sales_volume": ["avg_selling_price_eur", "gross_revenue_eur"],
    "fact_headcount":    ["avg_monthly_cost_per_fte_eur"],
}

tables = ["dim_account", "dim_date", "dim_department",
          "dim_product_line", "dim_scenario",
          "fact_gl_actuals", "fact_plan",
          "fact_sales_volume", "fact_headcount"]

for t in tables:
    # ---- everything indented below runs once per table ----

    cols = money_cols.get(t, [])   # money columns for this table, or []

    if len(cols) == 0:
        # No money columns: take every column as it is
        select_list = "*"
    else:
        # 1. Exclude the money columns from "*"
        select_list = "* EXCEPT (" + ", ".join(cols) + ")"
        # 2. Add each money column back, cast to DECIMAL
        for c in cols:
            select_list = select_list + ", CAST(" + c + " AS DECIMAL(18,2)) AS " + c

    # Create the empty target table (does nothing if it already exists)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {TARGET}.{t}")

    # Load the matching file
    result = spark.sql(f"""
        COPY INTO {TARGET}.{t}
        FROM (
          SELECT {select_list},
                 _metadata.file_path AS _source_file,
                 current_timestamp() AS _ingested_at
          FROM '{SOURCE}'
        )
        FILEFORMAT = CSV
        PATTERN = '{t}*.csv'
        FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
        COPY_OPTIONS  ('mergeSchema' = 'true')
    """)

    # Report: indented, so it prints for EVERY table
    n = result.first()["num_inserted_rows"]   # rows loaded by COPY INTO
    print(t, n, "rows inserted")

In [0]:
%sql
SELECT 'fact_gl_actuals' AS t, COUNT(*) FROM alpine.bronze.fact_gl_actuals
UNION ALL SELECT 'fact_plan', COUNT(*) FROM alpine.bronze.fact_plan
UNION ALL SELECT 'fact_sales_volume', COUNT(*) FROM alpine.bronze.fact_sales_volume
UNION ALL SELECT 'fact_headcount', COUNT(*) FROM alpine.bronze.fact_headcount;